# Session 1 · One nightly job, three ways

Queen City Trip Analytics, a fictional South End analytics company, rolls up one night of a fleet customer's trips before morning.

In [ ]:
source ~/dsba6190-live-demo-01/env.sh && echo "$BUCKET" \
  || echo "NOT STAGED. Run prep.ipynb first. Do not run any other cell."

## Step 1 · Who and where

In [ ]:
gcloud config list

In [ ]:
gcloud auth list --filter=status:ACTIVE

In [ ]:
gcloud projects describe "$PROJECT" --format="table(name,projectId,projectNumber,lifecycleState)"

## Step 2 · Where the project sits

In [ ]:
gcloud projects get-ancestors "$PROJECT"

In [ ]:
gcloud projects get-iam-policy "$PROJECT" --flatten="bindings[]" --format="table(bindings.role)"

## Step 3 · Geography

In [ ]:
gcloud compute regions list --format='value(name)' | wc -l
gcloud compute regions list --format='value(name)' | grep '^us-'

In [ ]:
gcloud compute zones list --filter="region:us-east1" --format="table(name,status)"

## Step 4 · A bucket, and what cannot change

In [ ]:
gcloud storage buckets create "gs://$SCRATCH" --location "$REGION" --uniform-bucket-level-access

In [ ]:
gcloud storage buckets describe "gs://$SCRATCH" --format="yaml(name,location,location_type,default_storage_class)"

In [ ]:
gcloud storage buckets update "gs://$SCRATCH" --location us

In [ ]:
gcloud storage buckets update "gs://$SCRATCH" --default-storage-class NEARLINE
gcloud storage buckets describe "gs://$SCRATCH" --format="value(default_storage_class)"

## Step 5 · Last night's file

In [ ]:
gcloud storage ls -l "gs://$BUCKET/raw/"

In [ ]:
gcloud storage cat -r 0-299 "gs://$BUCKET/raw/trips-2026-08-19.csv"

In [ ]:
cat job/rollup.sh

## Step 6 · The job on a virtual machine

In [ ]:
V0=$(date +%s)
gcloud compute instances create "$VM" --zone "$ZONE" --machine-type e2-standard-2 \
  --image-family debian-12 --image-project debian-cloud --scopes cloud-platform \
  --labels course=dsba6190,session=01 \
  --metadata startup-script="gcloud storage cat gs://$BUCKET/job/rollup.sh | bash -s $BUCKET vm; shutdown -h now"

In [ ]:
until [ "$(gcloud compute instances describe "$VM" --zone "$ZONE" --format='value(status)')" = TERMINATED ]; do sleep 10; done
VM_S=$(( $(date +%s) - V0 )); echo "create to TERMINATED: $VM_S s"
gcloud compute instances list --filter="name=$VM" --format="table(name,zone.basename(),machineType.basename(),status)"

In [ ]:
gcloud storage cat "gs://$BUCKET/out/vm/timing.txt"

In [ ]:
gcloud compute disks list --filter="name=$VM" --format="table(name,sizeGb,type.basename(),status,users.len():label=ATTACHED)"

## Step 7 · The same job on Cloud Run

In [ ]:
gcloud run jobs create "$JOB" --region "$REGION" --image gcr.io/google.com/cloudsdktool/google-cloud-cli:slim \
  --cpu 2 --memory 2Gi --max-retries 0 --task-timeout 15m --labels course=dsba6190,session=01 \
  --command bash --args="-c,gcloud storage cat gs://$BUCKET/job/rollup.sh | bash -s $BUCKET cloud-run"

In [ ]:
time gcloud run jobs execute "$JOB" --region "$REGION" --wait

In [ ]:
gcloud run jobs executions list --job "$JOB" --region "$REGION" \
  --format="table(name,status.succeededCount,status.startTime.date('%H:%M:%S'),status.completionTime.date('%H:%M:%S'))"

In [ ]:
gcloud storage cat "gs://$BUCKET/out/cloud-run/timing.txt"

## Step 8 · The same question in BigQuery

In [ ]:
bq --project_id="$PROJECT" --location=US mk --dataset "$PROJECT:$DATASET"

In [ ]:
time bq --project_id="$PROJECT" --location=US load --skip_leading_rows=1 "$DATASET.trips" "gs://$BUCKET/raw/trips-2026-08-19.csv" \
  trip_id:STRING,pickup_ts:DATETIME,pickup_zone:STRING,dropoff_zone:STRING,miles:NUMERIC,fare_usd:NUMERIC,vehicle_id:STRING

In [ ]:
q "SELECT pickup_zone, COUNT(*) AS trips, ROUND(SUM(fare_usd), 2) AS revenue_usd
   FROM \`$DATASET.trips\` GROUP BY 1 ORDER BY 1"

## Step 9 · Three platforms, one answer

In [ ]:
gcloud storage cat "gs://$BUCKET/out/vm/rollup.csv" > vm.csv
gcloud storage cat "gs://$BUCKET/out/cloud-run/rollup.csv" > run.csv
bq --project_id="$PROJECT" --location=US query --use_legacy_sql=false --format=csv --quiet \
  "SELECT pickup_zone, COUNT(*), FORMAT('%.2f', CAST(SUM(fare_usd) AS FLOAT64)) FROM \`$DATASET.trips\` GROUP BY 1 ORDER BY 1" \
  | tail -n +2 | tr -d '"' | sort > bq.csv
diff vm.csv run.csv && echo "VM and Cloud Run agree"
diff vm.csv bq.csv && echo "VM and BigQuery agree"

## Step 10 · What each run cost

In [ ]:
EXEC_S=$(gcloud run jobs executions list --job "$JOB" --region "$REGION" --format=json \
  | python3 -c 'import json,sys,datetime as d; e=json.load(sys.stdin)[0]["status"]; f=lambda s: d.datetime.fromisoformat(s.replace("Z","+00:00")); print(round((f(e["completionTime"])-f(e["startTime"])).total_seconds()))')
BQ_BYTES=$(bq --project_id="$PROJECT" --location=US --format=json show -j "$(cat .last_job)" | jq -r '.statistics.query.totalBytesBilled')
awk -v vm="$VM_S" -v cr="$EXEC_S" -v by="$BQ_BYTES" 'BEGIN {
  printf "%-20s %-26s %10s\n", "platform", "measured", "list price"
  printf "%-20s %-26s %10.5f\n", "Compute Engine", sprintf("%d s of e2-standard-2", vm), vm/3600*0.067006
  printf "%-20s %-26s %10.5f\n", "Cloud Run job", sprintf("%d s of 2 vCPU, 2 GiB", cr), cr*(2*0.000018 + 2*0.000002)
  printf "%-20s %-26s %10.5f\n", "BigQuery on-demand", sprintf("%.0f MB billed", by/1048576), by/1099511627776*6.25 }'

## Step 11 · Where cost appears

In [ ]:
gcloud billing projects describe "$PROJECT" --format="yaml(billingEnabled)"

In [ ]:
gcloud billing budgets list --billing-account="$(gcloud billing projects describe "$PROJECT" --format='value(billingAccountName.basename())')" \
  --format="yaml(displayName,amount,thresholdRules)"

## Step 12 · Teardown

In [ ]:
gcloud compute instances delete "${VM:?}" --zone "${ZONE:?}" --quiet
gcloud run jobs delete "${JOB:?}" --region "${REGION:?}" --quiet
bq --project_id="$PROJECT" rm -r -f -d "${DATASET:?}"
gcloud storage rm -r "gs://${BUCKET:?}" "gs://${SCRATCH:?}" --quiet

In [ ]:
gcloud compute instances list --filter='name~qc-nightly'
gcloud compute disks list --filter='name~qc-nightly'
gcloud run jobs list --region "$REGION"
gcloud storage ls | grep -c "$SUFFIX" || true